# Classes and inheritance 101
NOTE: this is a very basic introduction into classes, dundermethods and inheritance. If you are confused about classes and want to understand the basics of the concept a bit better, this is a nice place to start. 

## dataclass
Python has a native `dataclass` since 3.7
It is ideal to specify some data. Let's imagine we are starting a zoo:

In [ ]:
from dataclasses import dataclass


@dataclass
class Lion:
    food: str

We can now make an instance of the `Lion` class. Let's create a `Lion` named alex that eats `steak`

In [ ]:
alex = Lion(food="steak")

`alex` is now an object. It is an instance of the class `Lion`. The class specifies the general idea, in our case: a `Lion` is an object that has a single feature, which is `food` and `food` is a string. Obviously, that is very simple and basic, but we are trying to keep things as simple as possible for now.

In this specific case, we have `alex` and that is a `Lion` with a specific preference for food:

In [ ]:
alex.food

## the `__init__` method
the `@dataclass` wrapper is there to make life easier. It's the same as this:

In [ ]:
class Lion:
    def __init__(self, food: str) -> None:
        self.food = food

But that is a lot more [boilerplate](https://en.wikipedia.org/wiki/Boilerplate_code) code...

Now, we want to make our class more complex, such that we can also feed the `Lion`

In [ ]:
class Lion:
    def __init__(self, food: str) -> None:
        self.food = food

    def give_food(self):
        print(f"The lion eats the {self.food}")


leeuw = Lion(food="steak")
leeuw.give_food()

And, as our zoo is expanding, we add another lion

In [ ]:
fred = Lion(food="ham")
fred.give_food()

As you can see, we have two different animals with their own preferences, while both are `Lion`

In [ ]:
alex.food, fred.food

We go on expanding, and add an optimal time for feeding. Let's make it so that the optimal feeding time is generated at random at the moment the `Lion` is created. At the moment the `Lion` is created, the `__init__` method is always called. That is a one-time-event at the moment of initialization.

Once it is created, the properties of `Lion` stay the same (unless we actively change them)

In [ ]:
import numpy as np


class Lion:
    def __init__(self, food: str) -> None:
        self.food = food
        self.time: int = np.random.randint(9, 17)

    def give_food(self):
        print(f"The lion eats the {self.food}")

    def ideal_feeding_time(self) -> str:
        return f"{self.time}h"


alex = Lion(food="steak")

In [ ]:
alex.ideal_feeding_time()

## Adding optional parameters
Because we want to have the option to train the lion for a specific time of our own choosing, 
we add `time` as an `Optional` parameter with a `None` default. 

In [ ]:
from typing import Optional


class Lion:
    def __init__(self, food: str, time: Optional[int] = None) -> None:
        self.food = food
        if not time:
            time = np.random.randint(9, 17)
        self.time: int = time

    def give_food(self):
        print(f"The lion eats the {self.food}")

    def ideal_feeding_time(self) -> str:
        return f"{self.time}h"


alex = Lion(food="ham", time=8)
alex.ideal_feeding_time()

## More dunder methods
Let's add the dunder methods `__len__` and `__getitem__`, because `Lion` can now have multiple prefered foods.

> In Python, dunder methods are methods that allow instances of a class to interact with the built-in functions and operators of the language. The word “dunder” comes from “double underscore”, because the names of dunder methods start and end with two underscores, for example `__str__` or `__add__`. Typically, dunder methods are not invoked directly by the programmer. [source](https://mathspp.com/blog/pydonts/dunder-methods)

`__len__` returns the number of foods.
`__getitem__` returns a food by using and index `idx`

In [ ]:
from typing import List


class Lion:
    def __init__(self, food: List[str], time: Optional[int] = None) -> None:
        self.food = food
        if not time:
            time = np.random.randint(9, 17)
        self.time: int = time

    def give_food(self) -> None:
        print(f"The lion enjoys {len(self)} items")

    def __getitem__(self, idx: int) -> str:
        return self.food[idx]

    def __len__(self) -> int:
        return len(self.food)

    def ideal_feeding_time(self) -> str:
        return f"{self.time}h"


alex = Lion(["steak", "sushi"], time=9)
len(alex)

In [ ]:
alex.give_food()
alex[1]

So, what is happening here? The `__get_item__` method is called whenever you do `object[index]`, so in our case, when we create a `Lion` object `alex`, when we do `alex[1]` whatever is between the brackets is sent as an argument to the `__get_item__` method. We have specified that the argument is passed on to `self.food`.

The same thing is happening with `len`: when we call `len(object)`, under the hood the method `__len__` is called. We specified this for our `Lion` class at the return values of `len(self.food)`, but we could have defined it any way we like.


## Inheritance

Everything so far was one class standing on its own. The reason classes are worth knowing in
this course is what happens when there are two of them: a **parent** that knows the tedious
part, and a **child** that writes only the interesting one.

You will do this twice in the first two lessons, both times against a class from
`goad_toolkit`, and both times the child is about five lines. The rest of this notebook is
those two, because they are the ones you are about to meet.

### The one from lesson 1: `TransformBase`

A step in a data pipeline always does the same three things: check the column exists, do
something to the frame, hand the frame back. Only the middle one differs between steps.

`TransformBase` owns the other two. Look at its `__call__`:

```python
def __call__(self, data: pd.DataFrame) -> pd.DataFrame:
    self._validate_column(data)
    return self.transform(data, **self._params)
```

Whatever you passed to the constructor is kept in `self._params` and handed to `transform`.
So a step of your own is a class with one method:

In [ ]:
import pandas as pd
from goad_toolkit.datatransforms import TransformBase


class WordCount(TransformBase):
    """Add a column with the number of words in `column`."""

    def transform(self, data: pd.DataFrame, column: str) -> pd.DataFrame:
        data[f"{column}_words"] = data[column].str.split().str.len()
        return data


messages = pd.DataFrame({"message": ["hello there", "yes", "what time is the meeting"]})
WordCount(column="message")(messages)

Two things happened that you did not write.

`WordCount(column="message")` — no `__init__` anywhere in the class, so Python used
`TransformBase`'s, which stored `column="message"` and gave the step a `name`. And
`WordCount(...)(messages)` calls it like a function, because `__call__` is a dunder method
like the `__len__` and `__getitem__` above, and the parent defined it.

Ask for a column that is not there and the parent refuses before your code runs:

In [ ]:
try:
    WordCount(column="text")(messages)
except ValueError as error:
    print(f"ValueError: {error}")

### The one from lesson 2: `BasePlot`

The same shape, for charts. Every figure needs a figure and axes at the right size, a title,
axis labels and a grid — and then one line that actually draws something.

`BasePlot.plot()` does the first part and then calls `build()`, which is yours:

In [ ]:
import seaborn as sns
from goad_toolkit.visualizer import BasePlot, PlotSettings

from wa_analyzer.data import load_showcase


class BarPlot(BasePlot):
    """A bar chart. All the styling lives in PlotSettings."""

    def build(self, data: pd.DataFrame, x: str, y: str, **kwargs):
        sns.barplot(data=data, x=x, y=y, ax=self.ax, **kwargs)
        return self.fig, self.ax


settings = PlotSettings(figsize=(6, 3), title="Body mass by species",
                        xlabel="species", ylabel="body mass (g)")
fig, ax = BarPlot(settings).plot(data=load_showcase("penguins"), x="species", y="body_mass_g")

`self.fig` and `self.ax` exist by the time `build` runs, and the title and labels are already
on them, because `plot()` put them there first. Five lines of yours; the rest is inherited.

### What both of them are doing

The parent owns the *sequence* and the child owns one *step* of it. That pattern has a name —
the **template method** — and once you see it you will see it everywhere: `TransformBase`
validates then calls `transform`, `BasePlot` builds a figure then calls `build`, and in your
ML course `torch.nn.Module` does its bookkeeping then calls `forward`.

The parent also enforces the deal. Both `transform` and `build` are marked
`@abstractmethod`, which means a subclass that does not write one cannot be created at all:

In [ ]:
class Forgetful(TransformBase):
    pass


try:
    Forgetful()
except TypeError as error:
    print(f"TypeError: {error}")

That error is worth recognising, because it is the one you get when you subclass one of these
and misspell the method name — `def tranform(...)` leaves the abstract `transform`
unimplemented, and Python objects to the class rather than to the typo.

> **What to take to lesson 1.** A subclass is not a big commitment. It is a `class X(Parent):`
> line and one method, and everything else in this notebook — `__init__`, `__len__`,
> `__call__`, the lot — is already written for you in the parent.